# 01. Check CIFAR-10 k-shot subset

This notebook verifies that the generated split file `data/splits/cifar10/k20_seed0.json` works correctly.

Goals:

1. Load CIFAR-10.
2. Load the k-shot split file.
3. Show how many images are selected.
4. Count images per class.
5. Display a few selected images.


## 1. Imports and project paths

Run this notebook from the repository root or from the `notebooks/` folder.

In [ ]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
from torchvision.datasets import CIFAR10

# Detect project root.
# This allows the notebook to work whether you open it from the repo root or from notebooks/.
current_dir = Path.cwd()
if (current_dir / 'data').exists() and (current_dir / 'src').exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / 'data').exists() and (current_dir.parent / 'src').exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise RuntimeError('Could not find project root. Please run this notebook inside the SRP-Augmentation repository.')

DATA_ROOT = PROJECT_ROOT / 'data' / 'raw'
SPLIT_PATH = PROJECT_ROOT / 'data' / 'splits' / 'cifar10' / 'k20_seed0.json'
VAL_SPLIT_PATH = PROJECT_ROOT / 'data' / 'splits' / 'cifar10' / 'fixed_validation_split.json'

print('Project root:', PROJECT_ROOT)
print('Data root:', DATA_ROOT)
print('Split path:', SPLIT_PATH)

## 2. Load CIFAR-10

We load the original CIFAR-10 training dataset. The split file will tell us which images from this dataset belong to the small training subset.

In [ ]:
cifar10_train = CIFAR10(
    root=DATA_ROOT,
    train=True,
    download=True,
)

print('Number of original CIFAR-10 training images:', len(cifar10_train))
print('Classes:', cifar10_train.classes)

## 3. Load the k20 seed0 split

The JSON file does not contain images. It contains the indices of the selected images.

For CIFAR-10 with `k = 20`, we expect:

```text
10 classes × 20 images per class = 200 images
```

In [ ]:
with open(SPLIT_PATH, 'r') as f:
    split_info = json.load(f)

train_indices = split_info['train_indices']

print('Dataset:', split_info['dataset'])
print('k:', split_info['k'])
print('Subset seed:', split_info['subset_seed'])
print('Number of selected images:', len(train_indices))
print('First 10 selected indices:', train_indices[:10])

## 4. Count images per class

Because this is a stratified k-shot subset, each class should have exactly 20 images.

In [ ]:
targets = cifar10_train.targets
class_names = cifar10_train.classes

selected_labels = [targets[idx] for idx in train_indices]
class_counts = Counter(selected_labels)

class_count_table = pd.DataFrame({
    'class_id': list(range(len(class_names))),
    'class_name': class_names,
    'selected_count': [class_counts[i] for i in range(len(class_names))],
})

display(class_count_table)

In [ ]:
expected_k = split_info['k']
expected_total = expected_k * len(class_names)

assert len(train_indices) == expected_total, f'Expected {expected_total}, got {len(train_indices)}'
assert all(count == expected_k for count in class_counts.values()), 'Not all classes have exactly k images.'

print('Sanity check passed: the subset is balanced and has the expected size.')

## 5. Display a few selected images

This helps us visually confirm that the selected indices really correspond to CIFAR-10 images and labels.

In [ ]:
num_images_to_show = 20
sample_indices = train_indices[:num_images_to_show]

plt.figure(figsize=(12, 6))

for plot_position, dataset_idx in enumerate(sample_indices, start=1):
    image, label = cifar10_train[dataset_idx]

    plt.subplot(4, 5, plot_position)
    plt.imshow(image)
    plt.title(class_names[label])
    plt.axis('off')

plt.suptitle('First selected images from CIFAR-10 k20 seed0 subset')
plt.tight_layout()
plt.show()

## 6. Display examples grouped by class

This gives a better view of the stratified structure: each row shows images from one class.

In [ ]:
examples_per_class = 5

indices_by_class = {class_id: [] for class_id in range(len(class_names))}
for idx in train_indices:
    label = targets[idx]
    if len(indices_by_class[label]) < examples_per_class:
        indices_by_class[label].append(idx)

plt.figure(figsize=(12, 16))
plot_position = 1

for class_id, indices in indices_by_class.items():
    for dataset_idx in indices:
        image, label = cifar10_train[dataset_idx]
        plt.subplot(len(class_names), examples_per_class, plot_position)
        plt.imshow(image)
        if plot_position % examples_per_class == 1:
            plt.ylabel(class_names[class_id], rotation=0, labelpad=35, va='center')
        plt.xticks([])
        plt.yticks([])
        plot_position += 1

plt.suptitle('CIFAR-10 k20 seed0 subset: examples by class')
plt.tight_layout()
plt.show()